In [ ]:
# ── Harness setup ─────────────────────────────────────────────────────────────
import sys, os, json, pathlib
env_path = pathlib.Path('/home/elastic/env')
if env_path.exists():
    for line in env_path.read_text().splitlines():
        line = line.strip()
        if line and not line.startswith('#') and '=' in line:
            k, v = line.split('=', 1)
            os.environ.setdefault(k.strip(), v.strip())
from elasticsearch import Elasticsearch
es = Elasticsearch(os.environ['ES_URL'], api_key=os.environ['ES_API_KEY'])
print('Harness ready.')

In [ ]:
# ── Verify both indices have the same doc count ───────────────────────────────
for idx in ('cortex-corpus', 'cortex-corpus-candidate'):
    count = es.count(index=idx)['count']
    print(f'{idx}: {count} docs')

In [ ]:
# ── YOUR WORK ── Atomic alias cutover ────────────────────────────────────────
# Move cortex-corpus-live from cortex-corpus to cortex-corpus-candidate
# in ONE _aliases request (remove + add in the same actions list).
#
# Do NOT do this in two separate requests — the probe will detect the gap.

es.indices.update_aliases(body={
    'actions': [
        {'remove': {'index': 'cortex-corpus',           'alias': 'cortex-corpus-live'}},
        {'add':    {'index': 'cortex-corpus-candidate', 'alias': 'cortex-corpus-live'}},
    ]
})
print('Alias moved. cortex-corpus-live now points to cortex-corpus-candidate.')

# Verify
aliases = es.indices.get_alias(name='cortex-corpus-live')
print('Now points to:', list(aliases.keys()))